# Day 55：本地私有化推理服务（Ollama / vLLM OpenAI 兼容网关与吞吐压测）

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（微调与私有化部署）——**Day 55**！
> 今天我们将通过 `InferenceBackendConfig`、`OpenAICompatibleGateway`（基于 `httpx` + `MockTransport` 离线仿真）以及 `ThroughputBenchmarker`，完成本地 Ollama 与生产 vLLM 之间的零代码热切换，并实测对比两者的并发吞吐（TPS）、首字延迟（TTFT）与 P95 延迟。

## 1. 环境初始化与导入 `project_iterations/day_55` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / 'project_iterations' / 'day_55').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('未找到 project_iterations/day_55 目录')
    repo_root = repo_root.parent

day55_dir = repo_root / 'project_iterations' / 'day_55'
if str(day55_dir) not in sys.path:
    sys.path.insert(0, str(day55_dir))

from zhishu_agent import (
    InferenceBackendConfig,
    OpenAICompatibleGateway,
    ThroughputBenchmarker,
)

print('Day 55 模块加载成功:', day55_dir)

Day 55 模块加载成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_55


## 2. 配置抽象：Ollama vs vLLM 推理后端对比

In [2]:
ollama_cfg = InferenceBackendConfig.ollama_default()
vllm_cfg = InferenceBackendConfig.vllm_default()

for cfg in (ollama_cfg, vllm_cfg):
    print(
        f'[{cfg.backend_type:6s}] model={cfg.model_name} | '
        f'continuous_batching={cfg.supports_continuous_batching} | '
        f'kv={cfg.kv_cache_strategy}'
    )

[ollama] model=qwen2.5:14b-instruct-q4_K_M | continuous_batching=False | kv=Static Contiguous KV Allocation
[vllm  ] model=Qwen/Qwen2.5-14B-Instruct-AWQ | continuous_batching=True | kv=PagedAttention (Dynamic Block KV)


## 3. OpenAI 兼容网关演示：从 Ollama 热切换至 vLLM

In [3]:
messages = [{'role': 'user', 'content': '请生成查询近 7 天慢查询工单的只读 SQL'}]

with OpenAICompatibleGateway(ollama_cfg) as gateway:
    res_ollama = gateway.chat_completions(messages, max_tokens=96)
    print('Ollama 响应:', res_ollama.content)
    print(f'Ollama 指标: TTFT={res_ollama.ttft_ms} ms, tokens={res_ollama.completion_tokens}')

    # 热切换至 vLLM，业务调用代码零改动
    gateway.switch_backend(vllm_cfg)
    res_vllm = gateway.chat_completions(messages, max_tokens=96)
    print('\nvLLM 响应:', res_vllm.content)
    print(f'vLLM 指标: TTFT={res_vllm.ttft_ms} ms, tokens={res_vllm.completion_tokens}')

Ollama 响应: [OLLAMA::qwen2.5:14b-instruct-q4_K_M] 已收到智枢企业请求：'请生成查询近 7 天慢查询工单的只读 SQL'。当前 KV 策略：Static Contiguous KV Allocation。
Ollama 指标: TTFT=85.0 ms, tokens=35

vLLM 响应: [VLLM::Qwen/Qwen2.5-14B-Instruct-AWQ] 已收到智枢企业请求：'请生成查询近 7 天慢查询工单的只读 SQL'。当前 KV 策略：PagedAttention (Dynamic Block KV)。
vLLM 指标: TTFT=38.0 ms, tokens=35


## 4. 并发网关压测与 Ollama vs vLLM Continuous Batching 扩展性对比

In [4]:
prompts = [f'企业工单 #{i}：分析私有化部署显存瓶颈' for i in range(12)]

with OpenAICompatibleGateway(vllm_cfg) as vllm_gw:
    report = ThroughputBenchmarker.benchmark_gateway(
        vllm_gw, prompts=prompts, concurrency=4, max_tokens=96
    )
    print('=== vLLM 4 并发网关压测报告 ===')
    print(f'成功请求: {report.successful_requests}/{report.total_requests}')
    print(f'系统总吞吐: {report.system_tps:.1f} tok/s | TTFT(P95)={report.ttft_ms_p95:.2f} ms | 总延迟(P95)={report.latency_ms_p95:.2f} ms')

print('\n=== Ollama vs vLLM 多并发吞吐仿真对比 (output_tokens=128) ===')
print('并发数 | Ollama TTFT | Ollama TPS | vLLM TTFT | vLLM TPS  | 加速比')
for row in ThroughputBenchmarker.simulate_concurrency_comparison([1, 4, 8, 16, 32]):
    print(
        f"{row['concurrency']:>4d}   | {row['ollama_ttft_ms']:>8.1f} ms | "
        f"{row['ollama_system_tps']:>7.1f}/s | {row['vllm_ttft_ms']:>6.1f} ms | "
        f"{row['vllm_system_tps']:>7.1f}/s | {row['tps_speedup_vllm_vs_ollama']:>5.2f}x"
    )

=== vLLM 4 并发网关压测报告 ===
成功请求: 12/12
系统总吞吐: 53351.0 tok/s | TTFT(P95)=38.00 ms | 总延迟(P95)=38.00 ms

=== Ollama vs vLLM 多并发吞吐仿真对比 (output_tokens=128) ===
并发数 | Ollama TTFT | Ollama TPS | vLLM TTFT | vLLM TPS  | 加速比
   1   |     85.0 ms |    39.8/s |   42.5 ms |    64.1/s |  1.61x
   4   |     85.0 ms |   109.3/s |   48.4 ms |   246.0/s |  2.25x
   8   |   3886.3 ms |   125.1/s |   52.3 ms |   468.4/s |  3.74x
  16   |  11488.9 ms |   134.9/s |   56.4 ms |   862.9/s |  6.39x
  32   |  26694.1 ms |   140.4/s |   60.7 ms |  1543.9/s | 10.99x
